# 13 — Diagnóstico de sobreajuste (overfitting) dos modelos binários (RF e SVM)

**Pergunta do TCC:** os modelos que treinamos (RF padrão, RF ajustado, SVM padrão e SVM ajustado) sofreram de sobreajuste? Este notebook não treina nenhum modelo novo para uso: ele **mede sinais** nos modelos que já existem, sem alterar nenhum arquivo deles.

### Dois tipos de sobreajuste (e por que precisamos separar)
1. **Decorar o treino (clássico):** o modelo acerta as linhas que viu e erra linhas novas *do mesmo tipo*. A validação e a validação cruzada, que vêm do mesmo dia do treino, **detectam** isso.
2. **Ajustar-se ao dia do treino:** o modelo acerta dados do mesmo dia, mas cai em outro dia (o teste). A validação do mesmo dia **não detecta** isso; só o teste revela a queda, e a causa só aparece comparando os dados dos dois dias.

### O que o notebook faz
| Etapa | O que mede | Responde a |
|---|---|---|
| **2.1** Treino × validação cruzada × validação × teste | desempenho do modelo nas próprias linhas de treino, na CV, na validação e no teste (os números do teste vêm dos arquivos já gravados) | tipo 1 e tipo 2 |
| **3.1** Curva de aprendizagem | desempenho com 10%, 25%, 50%, 75% e 100% do ajuste | os dados bastam? a lacuna se fecha? |
| **4.1** Embaralhamento de rótulos | treina com rótulos embaralhados e avalia na validação real: o ROC-AUC deve ficar em ~0,5 | vazamento ou erro no processo |
| **5.1** Validação adversarial, por classe | um classificador tenta distinguir linhas do treino de linhas do teste, classe a classe; ROC-AUC ~0,5 = sem mudança, ~1 = os dois dias são diferentes | tipo 2: **o dado mudou?** e quais atributos |
| **6.1** Veredito | aplica as regras abaixo e escreve um texto para o TCC | conclusão |

### Regras de decisão (convenções, FIXADAS ANTES de olhar os resultados)
| Evidência | pequena / ok | moderada | grande / alerta |
|---|---|---|---|
| **Q1 — decorar o treino** | | | |
| lacuna treino − validação, acurácia balanceada (limiar natural) | < 0,01 | 0,01 a 0,03 | > 0,03 |
| lacuna treino − validação, ROC-AUC | < 0,005 | 0,005 a 0,02 | > 0,02 |
| embaralhamento de rótulos (ROC-AUC na validação real) | média das repetições dentro de 0,5 ± 0,03 **e** nenhuma repetição fora de 0,5 ± 0,10 | | alguma condição violada (investigar) |
| linhas da validação com vetor **idêntico** ao de uma linha do ajuste (etapa 4.2) | < 1% | 1% a 10% | > 10% (a validação do mesmo dia é otimista) |
| curva: ganho da validação de 50% para 100% dos dados (ROC-AUC) | < 0,0005 (platô) | 0,0005 a 0,003 | > 0,003 (ainda sobe: faltam dados) |
| **Q2 — generalizar para outro dia** | | | |
| queda validação → teste, ROC-AUC | < 0,005 | 0,005 a 0,02 | > 0,02 |
| razão do falso alarme, teste ÷ validação (limiar natural) | < 2× | 2× a 10× | > 10× |
| queda do recall, validação → teste (limiar natural) | < 0,01 | 0,01 a 0,05 | > 0,05 |
| validação adversarial: ROC-AUC treino × teste, por classe (classes com ao menos 200 linhas únicas no teste) | < 0,6 | 0,6 a 0,8 | > 0,8 |

**Veredito de Q1** = a pior gravidade entre as duas lacunas e o embaralhamento. **Veredito de Q2** = a pior gravidade entre as quatro linhas de Q2. A curva de aprendizagem é informativa e não entra no veredito.

### O que este notebook NÃO prova
- Ele **não prova** que não houve sobreajuste; mostra se **há sinais** e de que tipo. A conclusão do TCC deve ser escrita nesses termos.
- O teste continua sendo **uma única partição de outro dia**, com muitas linhas repetidas: as quedas são descritivas.
- Quando a validação do mesmo dia já está perto do teto (ROC-AUC ≈ 1), a lacuna treino − validação tem pouco espaço para aparecer; por isso o embaralhamento, a curva e a validação adversarial complementam a tabela.
- Os limites acima são convenções usuais, não leis. Por isso ficam gravados em `regras_decisao.json`, junto com os resultados.
- A validação adversarial usa os **atributos** (e o tipo de ataque, só para separar por classe) das linhas do teste, sem rótulo binário e sem calcular nenhuma métrica de modelo no teste; ela não decide nenhum hiperparâmetro. Fica declarado aqui.

### Custo esperado
Da ordem de 30 a 60 minutos no seu computador (a curva do SVM e a validação adversarial pesam mais). Os parâmetros opcionais `FAZER_CURVA`, `FAZER_PERM`, `FAZER_ADV`, `REPS_CURVA`, `N_ADV` permitem encurtar. Nada é apagado nem alterado nas pastas dos modelos.

### Correções desta versão (depois da primeira execução)
- **Embaralhamento:** o critério antigo ("4 desvios do acaso") gerava **alertas falsos**: um modelo treinado em rótulos embaralhados é uma função aleatória dos atributos e, por acaso, pode se relacionar com os rótulos reais; na prática, o ROC-AUC variou de 0,43 a 0,58 (média 0,50). O critério agora usa a **média das repetições**, com tolerância fixa. O texto sugerido do veredito foi corrigido de acordo.
- **Linhas repetidas entre ajuste e validação:** o embaralhamento **não** detecta isso (o modelo embaralhado não usa os rótulos reais). A nova etapa **4.2** conta diretamente os vetores idênticos.


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


## 0. Ferramentas
As mesmas dos notebooks 05, 08 e 12: conexão **somente leitura** com o ClickHouse (este notebook não cria nem apaga nada no servidor), leitura com espera quando o servidor fica sem memória, barra de progresso, o pré-processamento do RF/SVM (`PreprocessadorRF`), as métricas (`metricas`) e a previsão do SVM em vários processos (`prever_paralelo`).

In [6]:
import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

from joblib import Parallel, delayed

def _dec(modelo, X): return modelo.decision_function(X)
def prever_paralelo(modelo, X, k):
    # Escores (distância à fronteira) em k processos: o lote é dividido em k pedaços (o resultado é idêntico ao sequencial).
    if k <= 1: return modelo.decision_function(X)
    return np.concatenate(Parallel(n_jobs=k)(delayed(_dec)(modelo, c) for c in np.array_split(X, k)))

def medir_inferencia_svm(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, n_lote=20_000, repeticoes_lote=2, k_par=2):
    # Latência (1 fluxo por vez, 1 processo) e vazão (lote) com 1 e com k_par processos; com e sem o pré-processamento (mediana do treino + indicadores). O modelo inclui a padronização.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    for i in idx[:20]: modelo.decision_function(X_modelo[i:i + 1])                           # aquecimento
    t_mod, t_fim = [], []
    for i in idx:
        r = X_raw[i:i + 1]; t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.decision_function(m); t2 = time.perf_counter(); t_mod.append(t2 - t1); t_fim.append(t2 - t0)
    l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
    Xl, Xr = X_modelo[:n_lote], X_raw[:n_lote]; linhas = []
    for k in dict.fromkeys([1, max(int(k_par), 1)]):
        if k > 1: Parallel(n_jobs=k)(delayed(time.sleep)(0.01) for _ in range(k))
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); prever_paralelo(modelo, Xl, k); v_mod.append(len(Xl) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); prever_paralelo(modelo, prep.transformar(Xr), k); v_fim.append(len(Xr) / (time.perf_counter() - t0))
        nan = float("nan")
        linhas.append({"n_jobs": k, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0] if k == 1 else nan, "latencia_modelo_p95_ms": l_mod[1] if k == 1 else nan, "latencia_modelo_p99_ms": l_mod[2] if k == 1 else nan,
                       "latencia_fim_a_fim_p50_ms": l_fim[0] if k == 1 else nan, "latencia_fim_a_fim_p95_ms": l_fim[1] if k == 1 else nan, "latencia_fim_a_fim_p99_ms": l_fim[2] if k == 1 else nan,
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(Xl)})
    return linhas
print("Ferramentas prontas")

Autoteste das ferramentas do teste OK
Ferramentas prontas


## 1. Parâmetros, pastas, modelos e dados
O notebook usa os **modelos treinados só no ajuste** (`modelo_ajuste.joblib`; o padrão do SVM está em `modelos/C1_g1.joblib`), porque são eles que a validação separada nunca viu. Os modelos do refit (que também usaram a validação) são os do teste e não servem para medir a lacuna treino × validação. Os dados são os **mesmos** da busca: o ajuste e a validação do notebook 04. As versões da seleção e das views são conferidas contra as gravadas em cada modelo.

In [7]:
# ---------- 1.1 Parâmetros, pastas, modelos e dados ----------
import joblib, zlib, hashlib
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

REPO = Path(globals().get("REPO_ROOT", Path.cwd())); BASE = REPO / "artefatos_modelos"
SEMENTE = globals().get("SEMENTE", 42)
FAZER_CURVA, FAZER_PERM, FAZER_ADV = globals().get("FAZER_CURVA", True), globals().get("FAZER_PERM", True), globals().get("FAZER_ADV", True)
TAMANHOS_CURVA = tuple(globals().get("TAMANHOS_CURVA", (0.1, 0.25, 0.5, 0.75, 1.0))); REPS_CURVA = globals().get("REPS_CURVA", 3)
CURVA_PADRAO = globals().get("CURVA_PADRAO", True)                       # curva de aprendizagem também do RF padrão e do SVM padrão (o SVM padrão é bem mais lento)
N_PERM_RF, N_PERM_SVM = globals().get("N_PERM_RF", 5), globals().get("N_PERM_SVM", 3)
N_SVM_PERM, N_VAL_PERM_SVM = globals().get("N_SVM_PERM", 10_000), globals().get("N_VAL_PERM_SVM", 20_000)     # o SVM com rótulos embaralhados é lento: usa subamostras
MAX_ITER_PERM = globals().get("MAX_ITER_PERM", 2_000_000)                  # limite de iterações do SVM no embaralhamento (sem estrutura nos rótulos ele não converge)
N_ADV, ARVORES_ADV, K_ADV = globals().get("N_ADV", 20_000), globals().get("ARVORES_ADV", 100), globals().get("K_ADV", 5)
MIN_LINHAS_ADV = globals().get("MIN_LINHAS_ADV", 100)                      # abaixo disto a classe é pulada na validação adversarial
PROCESSOS_PREVISAO = globals().get("PROCESSOS_PREVISAO", 2)
T_SEC = {}                                                                  # tempo de cada etapa (gravado no fim)

# regras de decisão: fixadas ANTES de olhar qualquer resultado (as mesmas da introdução)
REGRAS = {"lacuna_acc_bal": (0.01, 0.03), "lacuna_roc_auc": (0.005, 0.02), "embaralhamento_media": 0.03, "embaralhamento_max": 0.10, "sobreposicao_validacao": (0.01, 0.10), "curva_ganho_50_100": (0.0005, 0.003),
          "queda_roc_auc_teste": (0.005, 0.02), "razao_falso_alarme": (2.0, 10.0), "queda_recall": (0.01, 0.05), "auc_adversarial": (0.6, 0.8), "min_linhas_unicas_adversarial": 200}

def ultima(base, padrao, exigir):
    ps = sorted(p for p in (BASE / base).glob(padrao) if all((p / f).exists() for f in exigir)); return ps[-1] if ps else None
PASTA_RF = globals().get("PASTA_RF") or ultima("rf_binario_busca", "rf_busca_*", ("modelo_ajuste.joblib", "FINALIZADO"))
PASTA_RF_PAD = (globals().get("PASTA_RF_PAD") or ultima("rf_binario", "rf_bin_*", ("modelo_ajuste.joblib",))) if globals().get("INCLUIR_RF_PADRAO", True) else None
PASTA_SVM = globals().get("PASTA_SVM") or ultima("svm_binario_busca", "svm_busca_*", ("modelo_ajuste.joblib", "FINALIZADO"))
assert PASTA_RF and PASTA_SVM, "faltam as pastas dos modelos ajustados (notebooks 07 e 11 concluídos)"
PASTA_RF, PASTA_SVM = Path(PASTA_RF), Path(PASTA_SVM); PASTA_RF_PAD = Path(PASTA_RF_PAD) if PASTA_RF_PAD else None
if PASTA_RF_PAD is None: print("⚠ " + ("RF padrão excluído por INCLUIR_RF_PADRAO = False." if not globals().get("INCLUIR_RF_PADRAO", True) else "modelo padrão do RF (notebook 05) não encontrado: ele fica fora do diagnóstico"))

V_AJUSTE, V_VALID, SEL_TAB = "prep_vw_bal_bin_treino", "prep_vw_bal_bin_validacao", "prep_bal_bin_selecao"
VIEW_TREINO, VIEW_TESTE, TAB_TREINO, TAB_TESTE = "prep_vw_rf_bin_treino", "prep_vw_rf_bin_teste_sem_sobrep", "prep_final_treino", "prep_final_teste"
faltam = [t for t in (TAB_TREINO, TAB_TESTE, V_AJUSTE, V_VALID, SEL_TAB, VIEW_TREINO, VIEW_TESTE, "prep_views_rf_binario") if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03 e 04: faltam {faltam}"
versao_sel = str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])
aud = consultar_num("SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM prep_views_rf_binario ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])

# ---- dados (iguais aos da busca): ajuste e validação, na mesma ordem ----
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views); CM = prep.colunas_modelo
assert CM == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")] == [c for c in colunas_view(VIEW_TESTE) if c not in ("id_linha", "y")], "a ordem das colunas das views difere"
def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID))
X_aj, y_aj = aj[CM].to_numpy(float), aj["y"].to_numpy(int); X_va, y_va = va[CM].to_numpy(float), va["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
nome_tipo = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
sel = consultar(f"SELECT toString(id_linha) AS id_linha, papel, tipo FROM {SEL_TAB}"); sel["tipo"] = sel["tipo"].map(nome_tipo)
tipo_aj = aj[["id_linha"]].merge(sel[sel.papel == "treino"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]
tipo_va = va[["id_linha"]].merge(sel[sel.papel == "validacao"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]
assert tipo_aj.notna().all() and tipo_va.notna().all(), "há linhas sem tipo de ataque na tabela de seleção"
TIPOS = ["BENIGN"] + sorted(t for t in set(tipo_aj) | set(tipo_va) if t != "BENIGN"); COD = {t: i for i, t in enumerate(TIPOS)}
g_aj, g_va = tipo_aj.map(COD).to_numpy(np.int16), tipo_va.map(COD).to_numpy(np.int16)
assert ((g_aj == 0) == (y_aj == 0)).all() and ((g_va == 0) == (y_va == 0)).all(), "o tipo BENIGN não coincide com y = 0"
assert not set(aj["id_linha"]) & set(va["id_linha"]), "ajuste e validação compartilham linhas"

# ---- modelos (treinados só no ajuste) ----
def pontuar_rf(m, X): return m.predict_proba(X)[:, 1]
def pontuar_svm(m, X): return prever_paralelo(m, X, PROCESSOS_PREVISAO)
def cfg_de(pasta): return json.loads((pasta / "config_treino.json").read_text(encoding="utf-8")) if (pasta / "config_treino.json").exists() else {}
MODELOS = {}
def registrar(chave, rotulo, familia, arquivo, pasta_cfg, pontuar, limiar, pasta_teste, col_teste, cv_fonte, descricao):
    m = joblib.load(arquivo)
    if familia == "RF": assert list(getattr(m, "colunas_modelo_", CM)) == CM, f"{rotulo}: as colunas do modelo diferem das das views"
    cfg = cfg_de(pasta_cfg)
    for k, v in (("versao_selecao_notebook04", versao_sel), ("versao_views_notebook03", versao_views)):
        if cfg.get(k) not in (None, v): print(f"⚠ {rotulo}: {k} do modelo = {cfg.get(k)} difere da dos dados atuais ({v})")
    MODELOS[chave] = dict(rotulo=rotulo, familia=familia, modelo=m, pontuar=pontuar, limiar=limiar, pasta=pasta_cfg, pasta_teste=pasta_teste, col_teste=col_teste, cv_fonte=cv_fonte, descricao=descricao, cfg=cfg)
cfg_rf, cfg_svm = cfg_de(PASTA_RF), cfg_de(PASTA_SVM)
hp_rf, hp_svm = cfg_rf.get("hiperparametros", {}), cfg_svm.get("hiperparametros", {})
registrar("rf_ajustado", "RF ajustado", "RF", PASTA_RF / "modelo_ajuste.joblib", PASTA_RF, pontuar_rf, 0.5, PASTA_RF, 1, ("json", PASTA_RF), f"{hp_rf.get('n_estimators')} árvores | profundidade {hp_rf.get('max_depth')} | folha {hp_rf.get('min_samples_leaf')}")
if PASTA_RF_PAD is not None: registrar("rf_padrao", "RF padrão", "RF", PASTA_RF_PAD / "modelo_ajuste.joblib", PASTA_RF_PAD, pontuar_rf, 0.5, PASTA_RF, 2, ("json", PASTA_RF_PAD), "100 árvores | profundidade livre | folha 1")
registrar("svm_ajustado", "SVM ajustado", "SVM", PASTA_SVM / "modelo_ajuste.joblib", PASTA_SVM, pontuar_svm, float(cfg_svm.get("limiar_padrao", 0.0)), PASTA_SVM, 1, ("json", PASTA_SVM), f"C = {hp_svm.get('C')} | gamma = {float(hp_svm.get('gamma', float('nan'))):.5f} ({hp_svm.get('gamma_fator')}× o de referência)")
arq_pad_svm = PASTA_SVM / "modelos" / "C1_g1.joblib"
if arq_pad_svm.exists(): registrar("svm_padrao", "SVM padrão", "SVM", arq_pad_svm, PASTA_SVM, pontuar_svm, float(cfg_svm.get("limiar_padrao", 0.0)), PASTA_SVM, 2, ("grade", PASTA_SVM, "C=1|gamma=1x"), "C = 1 | gamma = 1× o de referência")
else: print("⚠ o SVM padrão (modelos/C1_g1.joblib) não foi encontrado: fica fora do diagnóstico")

print(f"Seleção do notebook 04 (versão {versao_sel}) | views do notebook 03 (versão {versao_views})")
print(f"Ajuste: {len(X_aj):,} linhas ({int((y_aj == 0).sum()):,} BENIGN) | validação: {len(X_va):,} ({int((y_va == 0).sum()):,} BENIGN) | {len(TIPOS) - 1} tipos de ataque | {len(CM)} atributos")
print("\nModelos (treinados só no ajuste):")
for k, M in MODELOS.items(): print(f"  {M['rotulo']:13s} limiar natural {M['limiar']:g} | {M['descricao']} | {M['pasta'].name}")

# pasta do diagnóstico e GRAVAÇÃO IMEDIATA das regras de decisão (antes de calcular qualquer resultado)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA_DIAG = BASE / "diagnostico_sobreajuste" / f"diag_{RUN_ID}"; PASTA_DIAG.mkdir(parents=True, exist_ok=True)
salvar_json({"gravado_em": datetime.now().isoformat(timespec="seconds"), "observacao": "regras fixadas ANTES de calcular qualquer resultado deste diagnóstico (são as da introdução do notebook)", "regras": REGRAS}, PASTA_DIAG / "regras_decisao.json")
print(f"\nPasta do diagnóstico: {PASTA_DIAG}")

Seleção do notebook 04 (versão 20261018_204854) | views do notebook 03 (versão 20261018_180729)
Ajuste: 81,228 linhas (40,614 BENIGN) | validação: 60,154 (10,154 BENIGN) | 12 tipos de ataque | 43 atributos

Modelos (treinados só no ajuste):
  RF ajustado   limiar natural 0.5 | 300 árvores | profundidade 15 | folha 1 | rf_busca_20261024_175105
  RF padrão     limiar natural 0.5 | 100 árvores | profundidade livre | folha 1 | rf_bin_20261021_221311
  SVM ajustado  limiar natural 0 | C = 100.0 | gamma = 0.09521 (4.0× o de referência) | svm_busca_20261025_004828
  SVM padrão    limiar natural 0 | C = 1 | gamma = 1× o de referência | svm_busca_20261025_004828

Pasta do diagnóstico: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\diagnostico_sobreajuste\diag_20261111_230952


## 2. Treino × validação cruzada × validação × teste (tipo 1 e tipo 2)
Para cada modelo, as mesmas métricas **que não dependem da proporção de BENIGN** (ROC-AUC, acurácia balanceada, recall e falso alarme) em quatro lugares:
- **treino:** nas próprias linhas do ajuste (o que o modelo "decorou", se decorou);
- **CV:** média das 5 dobras da busca (linhas que o modelo não viu, do mesmo dia);
- **validação:** o conjunto separado do notebook 04 (também do mesmo dia);
- **teste:** os números **já gravados** pelos notebooks 08 e 12 (nenhuma previsão nova no teste).

A **lacuna treino − validação** mede o tipo 1. A **queda validação → teste** mede o tipo 2. Um RF sem limite de profundidade acerta ~100% do treino mesmo generalizando bem; por isso a lacuna é comparada com a CV e a validação, não lida sozinha. O SVM não guarda as linhas, mas um C e um gamma muito altos poderiam ajustar ruído: o mesmo teste se aplica.

In [8]:
# ---------- 2.1 Treino × CV × validação × teste ----------
t_sec0 = time.perf_counter()
def basicas(y, s, lim):
    m = metricas(y, s, lim); return {"roc_auc": m["roc_auc"], "acc_bal": m["acuracia_balanceada"], "recall": m["recall"], "fpr": m["taxa_falso_alarme"]}
def sev(v, lims):
    if v is None or not np.isfinite(v): return "n/d"
    return "pequena" if v < lims[0] else ("moderada" if v <= lims[1] else "grande")
ORDEM_SEV = {"n/d": -1, "pequena": 0, "moderada": 1, "grande": 2}
def pior(*sevs): return max(sevs, key=lambda s: ORDEM_SEV[s]) if sevs else "n/d"

def cv_de(M):
    nan = {"roc_auc": np.nan, "recall": np.nan, "fpr": np.nan, "acc_bal": np.nan}
    try:
        if M["cv_fonte"][0] == "json":
            r = json.loads((M["cv_fonte"][1] / "metricas_validacao.json").read_text(encoding="utf-8"))["cv_ajuste"]["resumo"]
            out = {"roc_auc": r["roc_auc"]["mean"], "recall": r["recall"]["mean"], "fpr": r["taxa_falso_alarme"]["mean"]}
        else:
            g = pd.read_csv(M["cv_fonte"][1] / "resultados_grade.csv"); g = g[g["config"] == M["cv_fonte"][2]].iloc[0]
            out = {"roc_auc": float(g["cv_roc_auc"]), "recall": float(g["cv_recall"]), "fpr": float(g["cv_taxa_falso_alarme"])}
        out["acc_bal"] = (out["recall"] + 1 - out["fpr"]) / 2; return out
    except Exception as e:
        print(f"⚠ CV de {M['rotulo']} não lida ({type(e).__name__}); coluna fica vazia"); return nan
def teste_de(M):
    nan = {"roc_auc": np.nan, "recall": np.nan, "fpr": np.nan, "acc_bal": np.nan}; arq = M["pasta_teste"] / "comparacao_padrao_vs_ajustado_teste.csv"
    try:
        d = pd.read_csv(arq); col = d.columns[M["col_teste"]]; v = d.set_index(d.columns[0])[col]
        out = {"roc_auc": float(v["roc_auc"]), "recall": float(v["recall"]), "fpr": float(v["taxa_falso_alarme"])}; out["acc_bal"] = (out["recall"] + 1 - out["fpr"]) / 2; return out
    except Exception as e:
        print(f"⚠ teste de {M['rotulo']} não lido de {arq.name} ({type(e).__name__}); coluna fica vazia"); return nan

ESC, linhas = {}, []
barra = Barra(len(MODELOS), "Pontuando ajuste e validação")
for chave, M in MODELOS.items():
    t0 = time.perf_counter(); s_aj, s_va = M["pontuar"](M["modelo"], X_aj), M["pontuar"](M["modelo"], X_va)
    assert np.isfinite(s_aj).all() and np.isfinite(s_va).all(), f"escores não finitos em {M['rotulo']}"
    ESC[chave] = {"aj": s_aj, "va": s_va}
    tr, vv, cv, te = basicas(y_aj, s_aj, M["limiar"]), basicas(y_va, s_va, M["limiar"]), cv_de(M), teste_de(M)
    r = {"modelo": M["rotulo"], "limiar_natural": M["limiar"]}
    for pref, d in (("treino", tr), ("cv", cv), ("validacao", vv), ("teste", te)):
        for k, v in d.items(): r[f"{pref}_{k}"] = v
    r["lacuna_acc_bal"] = tr["acc_bal"] - vv["acc_bal"]; r["lacuna_roc_auc"] = tr["roc_auc"] - vv["roc_auc"]
    r["queda_roc_auc_teste"] = vv["roc_auc"] - te["roc_auc"]; r["razao_falso_alarme"] = (te["fpr"] / vv["fpr"]) if vv["fpr"] > 0 else (np.inf if te["fpr"] > 0 else np.nan); r["queda_recall"] = vv["recall"] - te["recall"]
    linhas.append(r); barra.avancar(1, f"{M['rotulo']}: {time.perf_counter() - t0:.0f} s")
barra.fechar()
TAB = pd.DataFrame(linhas).set_index("modelo")
print("\n== Métricas por origem (limiar natural: RF 0,5 | SVM 0) ==")
display(TAB[[c for c in TAB.columns if c.split("_")[0] in ("treino", "cv", "validacao", "teste") and not c.endswith("fpr")]].round(5))
print("\n== Lacunas e quedas ==")
LAC = TAB[["lacuna_acc_bal", "lacuna_roc_auc", "queda_roc_auc_teste", "razao_falso_alarme", "queda_recall"]].copy()
LAC["Q1 acc.bal."] = [sev(max(v, 0), REGRAS["lacuna_acc_bal"]) for v in TAB.lacuna_acc_bal]; LAC["Q1 ROC-AUC"] = [sev(max(v, 0), REGRAS["lacuna_roc_auc"]) for v in TAB.lacuna_roc_auc]
LAC["Q2 ROC-AUC"] = [sev(v, REGRAS["queda_roc_auc_teste"]) for v in TAB.queda_roc_auc_teste]; LAC["Q2 falso alarme"] = [sev(v, REGRAS["razao_falso_alarme"]) for v in TAB.razao_falso_alarme]
LAC["Q2 recall"] = [sev(max(v, 0) if np.isfinite(v) else v, REGRAS["queda_recall"]) for v in TAB.queda_recall]
display(LAC.round(5))
T_SEC["2.1_tabela"] = time.perf_counter() - t_sec0

Pontuando ajuste e validação: concluído em 58s

== Métricas por origem (limiar natural: RF 0,5 | SVM 0) ==


,treino_roc_auc,treino_acc_bal,treino_recall,cv_roc_auc,cv_recall,cv_acc_bal,validacao_roc_auc,validacao_acc_bal,validacao_recall,teste_roc_auc,teste_recall,teste_acc_bal
modelo,,,,,,,,,,,,
RF ajustado,1.00000,0.99991,0.99993,0.99999,0.99897,0.99897,0.99999,0.99920,0.99918,0.99136,0.95259,0.96748
RF padrão,1.00000,0.99998,0.99995,0.99996,0.99879,0.99890,0.99994,0.99922,0.99922,0.98993,0.95111,0.96508
SVM ajustado,0.99961,0.99625,0.99483,0.99917,0.99372,0.99551,0.99948,0.99698,0.99622,0.99531,0.99642,0.99241
SVM padrão,0.99910,0.99275,0.98978,0.99889,0.98966,0.99260,0.99930,0.99491,0.99406,0.99504,0.74595,0.86842



== Lacunas e quedas ==


,lacuna_acc_bal,lacuna_roc_auc,queda_roc_auc_teste,razao_falso_alarme,queda_recall,Q1 acc.bal.,Q1 ROC-AUC,Q2 ROC-AUC,Q2 falso alarme,Q2 recall
modelo,,,,,,,,,,
RF ajustado,0.00072,0.00001,0.00864,22.37489,0.04659,pequena,pequena,moderada,grande,moderada
RF padrão,0.00076,0.00006,0.01000,26.59699,0.04811,pequena,pequena,moderada,grande,moderada
SVM ajustado,-0.00073,0.00013,0.00418,5.12032,-0.00020,pequena,pequena,pequena,moderada,pequena
SVM padrão,-0.00216,-0.00020,0.00425,2.15069,0.24811,pequena,pequena,pequena,moderada,grande


## 3. Curva de aprendizagem
O modelo é treinado de novo com **10%, 25%, 50%, 75% e 100%** das linhas do ajuste (subamostras estratificadas por tipo de ataque; 3 sorteios para cada fração abaixo de 100%) e avaliado **na mesma validação** de sempre. Em cada ponto aparecem o desempenho **nas linhas de treino daquele tamanho** e **na validação**.
- Treino ≫ validação que **não se fecha** com mais dados: sinal de sobreajuste (variância).
- Validação que **ainda sobe** de 50% para 100%: o modelo está limitado por dados, não por complexidade.
- Os dois **juntos e estáveis**: o modelo já extraiu o que os dados permitem.

O ponto de 100% é o modelo treinado do jeito original: seu resultado precisa **reproduzir** o gravado (checagem final).

In [9]:
# ---------- 3.1 Curva de aprendizagem ----------
t_sec0 = time.perf_counter()
CURVA, CURVA_RES, GANHO_50_100, FIG_CURVA = pd.DataFrame(), pd.DataFrame(), {}, None
def sub_estrat(g, frac, semente, minimo=20):
    rng = np.random.default_rng(semente); idx = []
    for t in np.unique(g):
        pos = np.flatnonzero(g == t); n = len(pos) if frac >= 1 else min(len(pos), max(int(np.ceil(frac * len(pos))), minimo))
        idx.append(rng.choice(pos, size=n, replace=False) if n < len(pos) else pos)
    return np.sort(np.concatenate(idx))
if FAZER_CURVA:
    quem = [k for k in MODELOS if k.endswith("ajustado") or CURVA_PADRAO]
    n_fits = sum((1 if f >= 1 else REPS_CURVA) for f in TAMANHOS_CURVA) * len(quem); barra = Barra(n_fits, "Curva de aprendizagem"); linhas = []
    for k in quem:
        M = MODELOS[k]
        for frac in TAMANHOS_CURVA:
            for r in range(1 if frac >= 1 else REPS_CURVA):
                idx = sub_estrat(g_aj, frac, SEMENTE + 1000 * r + int(round(frac * 100)))
                t0 = time.perf_counter(); est = clone(M["modelo"]).fit(X_aj[idx], y_aj[idx]); t_fit = time.perf_counter() - t0
                a, b = basicas(y_aj[idx], M["pontuar"](est, X_aj[idx]), M["limiar"]), basicas(y_va, M["pontuar"](est, X_va), M["limiar"])
                linhas.append({"modelo": M["rotulo"], "fracao": frac, "sorteio": r, "n_linhas": len(idx), "treino_roc_auc": a["roc_auc"], "treino_acc_bal": a["acc_bal"], "validacao_roc_auc": b["roc_auc"], "validacao_acc_bal": b["acc_bal"],
                               "validacao_recall": b["recall"], "validacao_fpr": b["fpr"], "tempo_treino_s": t_fit})
                barra.avancar(1, f"{M['rotulo']} | {int(100 * frac)}% ({len(idx):,} linhas) | treino {t_fit:.0f} s")
    barra.fechar(); CURVA = pd.DataFrame(linhas)
    CURVA_RES = CURVA.groupby(["modelo", "fracao"]).agg(n_linhas=("n_linhas", "first"), treino_roc_auc=("treino_roc_auc", "mean"), validacao_roc_auc=("validacao_roc_auc", "mean"), validacao_roc_auc_dp=("validacao_roc_auc", "std"),
                                                         treino_acc_bal=("treino_acc_bal", "mean"), validacao_acc_bal=("validacao_acc_bal", "mean"), tempo_treino_s=("tempo_treino_s", "mean")).reset_index()
    CURVA_RES["lacuna_acc_bal"] = CURVA_RES.treino_acc_bal - CURVA_RES.validacao_acc_bal; CURVA_RES["lacuna_roc_auc"] = CURVA_RES.treino_roc_auc - CURVA_RES.validacao_roc_auc
    print("\n== Curva de aprendizagem (média dos sorteios; dp = desvio da validação entre sorteios) ==")
    display(CURVA_RES.round(5))
    for rot, d in CURVA_RES.groupby("modelo"):
        d = d.set_index("fracao")
        if 0.5 in d.index and 1.0 in d.index:
            GANHO_50_100[rot] = float(d.loc[1.0, "validacao_roc_auc"] - d.loc[0.5, "validacao_roc_auc"])
            print(f"  {rot}: ganho da validação de 50% para 100% dos dados = {GANHO_50_100[rot]:+.5f} ROC-AUC → {sev(GANHO_50_100[rot], REGRAS['curva_ganho_50_100']).replace('pequena', 'platô').replace('moderada', 'sobe pouco').replace('grande', 'ainda sobe')} | lacuna acc. balanceada em 100%: {d.loc[1.0, 'lacuna_acc_bal']:+.5f}")
    try:
        import matplotlib.pyplot as plt
        rots = list(CURVA_RES.modelo.unique()); FIG_CURVA, ax = plt.subplots(len(rots), 2, figsize=(11, 3.6 * len(rots)), squeeze=False)
        for i, rot in enumerate(rots):
            d = CURVA_RES[CURVA_RES.modelo == rot]
            for j, (c_tr, c_va, tit) in enumerate((("treino_roc_auc", "validacao_roc_auc", "ROC-AUC"), ("treino_acc_bal", "validacao_acc_bal", "acurácia balanceada (limiar natural)"))):
                a_ = ax[i, j]; a_.plot(d.n_linhas, d[c_tr], "o--", label="treino (linhas usadas)"); a_.plot(d.n_linhas, d[c_va], "o-", label="validação")
                if j == 0 and d.validacao_roc_auc_dp.notna().any(): a_.errorbar(d.n_linhas, d[c_va], yerr=d.validacao_roc_auc_dp.fillna(0), fmt="none", capsize=3, color="C1")
                a_.set_title(f"{rot}: {tit}"); a_.set_xlabel("linhas de treino"); a_.grid(alpha=0.3)
                if i == 0 and j == 0: a_.legend()
        FIG_CURVA.suptitle("Curva de aprendizagem: treino × validação"); FIG_CURVA.tight_layout(); plt.close(FIG_CURVA)
    except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")
else: print("Curva de aprendizagem desligada (FAZER_CURVA = False).")
T_SEC["3.1_curva"] = time.perf_counter() - t_sec0

Curva de aprendizagem: concluído em 14min 29s

== Curva de aprendizagem (média dos sorteios; dp = desvio da validação entre sorteios) ==


,modelo,fracao,n_linhas,treino_roc_auc,validacao_roc_auc,validacao_roc_auc_dp,treino_acc_bal,validacao_acc_bal,tempo_treino_s,lacuna_acc_bal,lacuna_roc_auc
0,RF ajustado,0.10,8133,1.00000,0.99996,0.00003,1.00000,0.99776,1.16078,0.00224,0.00004
1,RF ajustado,0.25,20313,1.00000,0.99999,0.00000,1.00000,0.99871,1.30408,0.00129,0.00001
2,RF ajustado,0.50,40615,1.00000,0.99999,0.00000,0.99998,0.99903,4.65424,0.00095,0.00001
3,RF ajustado,0.75,60928,1.00000,0.99999,0.00000,0.99993,0.99904,8.45845,0.00089,0.00001
4,RF ajustado,1.00,81228,1.00000,0.99999,NaN,0.99991,0.99920,15.20827,0.00072,0.00001
5,RF padrão,0.10,8133,1.00000,0.99995,0.00002,1.00000,0.99775,0.42350,0.00225,0.00005
6,RF padrão,0.25,20313,1.00000,0.99996,0.00003,1.00000,0.99869,0.53479,0.00131,0.00004
7,RF padrão,0.50,40615,1.00000,0.99998,0.00001,0.99999,0.99907,1.19738,0.00092,0.00002
8,RF padrão,0.75,60928,1.00000,0.99999,0.00001,0.99998,0.99912,2.82037,0.00085,0.00001
9,RF padrão,1.00,81228,1.00000,0.99994,NaN,0.99998,0.99922,5.04523,0.00076,0.00006


  RF ajustado: ganho da validação de 50% para 100% dos dados = +0.00000 ROC-AUC → platô | lacuna acc. balanceada em 100%: +0.00072
  RF padrão: ganho da validação de 50% para 100% dos dados = -0.00005 ROC-AUC → platô | lacuna acc. balanceada em 100%: +0.00076
  SVM ajustado: ganho da validação de 50% para 100% dos dados = +0.00007 ROC-AUC → platô | lacuna acc. balanceada em 100%: -0.00073
  SVM padrão: ganho da validação de 50% para 100% dos dados = +0.00008 ROC-AUC → platô | lacuna acc. balanceada em 100%: -0.00216


## 4. Embaralhamento de rótulos (controle negativo)
O rótulo (BENIGN × ataque) das linhas do ajuste é **embaralhado ao acaso**, o modelo é treinado assim e avaliado **na validação com os rótulos reais**. Sem nenhuma relação entre atributos e rótulo, o ROC-AUC deve ficar em torno de **0,5**. Como o modelo treinado é uma função aleatória dos atributos, **cada repetição pode se afastar de 0,5** (vimos de 0,43 a 0,58); por isso a regra usa a **média das repetições** (0,5 ± 0,03) e só alerta se alguma repetição passar de ± 0,10.

**O que este controle NÃO detecta:** linhas repetidas entre ajuste e validação (o modelo embaralhado não usa os rótulos reais). Isso é medido na etapa **4.2**.

O RF usa o ajuste inteiro. O **SVM usa uma subamostra** (por padrão, 10 mil linhas do ajuste e 20 mil da validação), porque com rótulos sem estrutura quase toda linha vira vetor de suporte e o treino fica muito lento; o número de iterações é limitado, o que não importa para um modelo que não deve aprender nada.

In [10]:
# ---------- 4.1 Embaralhamento de rótulos ----------
import warnings
t_sec0 = time.perf_counter(); PERM, PERM_RES = pd.DataFrame(), pd.DataFrame()
if FAZER_PERM:
    quem = [k for k in MODELOS if k.endswith("ajustado")]; linhas = []
    barra = Barra(sum(N_PERM_RF if MODELOS[k]["familia"] == "RF" else N_PERM_SVM for k in quem), "Embaralhamento de rótulos")
    for k in quem:
        M = MODELOS[k]; eh_rf = M["familia"] == "RF"; n_rep = N_PERM_RF if eh_rf else N_PERM_SVM
        if eh_rf: idx_fit, idx_ev = np.arange(len(y_aj)), np.arange(len(y_va))
        else: idx_fit, idx_ev = sub_estrat(g_aj, min(1.0, N_SVM_PERM / len(y_aj)), SEMENTE), sub_estrat(g_va, min(1.0, N_VAL_PERM_SVM / len(y_va)), SEMENTE)
        for r in range(n_rep):
            y_emb = np.random.default_rng(SEMENTE + 7 * r + 1).permutation(y_aj[idx_fit]); est = clone(M["modelo"])
            if not eh_rf: est.set_params(svm__max_iter=MAX_ITER_PERM)
            t0 = time.perf_counter()
            with warnings.catch_warnings(): warnings.simplefilter("ignore"); est.fit(X_aj[idx_fit], y_emb)
            auc = float(roc_auc_score(y_va[idx_ev], M["pontuar"](est, X_va[idx_ev])))
            linhas.append({"modelo": M["rotulo"], "repeticao": r, "n_treino": len(idx_fit), "n_avaliacao": len(idx_ev), "roc_auc_embaralhado": auc, "desvio_de_0_5": auc - 0.5, "concordancia_com_rotulo_real": float((y_emb == y_aj[idx_fit]).mean()), "tempo_s": time.perf_counter() - t0})
            barra.avancar(1, f"{M['rotulo']} | repetição {r + 1}/{n_rep} | ROC-AUC {auc:.4f} | {time.perf_counter() - t0:.0f} s")
    barra.fechar(); PERM = pd.DataFrame(linhas); PERM["desvio_abs"] = PERM.desvio_de_0_5.abs()
    PERM_RES = PERM.groupby("modelo").agg(repeticoes=("roc_auc_embaralhado", "size"), roc_auc_medio=("roc_auc_embaralhado", "mean"), roc_auc_min=("roc_auc_embaralhado", "min"), roc_auc_max=("roc_auc_embaralhado", "max"), max_desvio_abs=("desvio_abs", "max"), n_treino=("n_treino", "first"), n_avaliacao=("n_avaliacao", "first")).reset_index()
    PERM_RES["situacao"] = ["ok (média a menos de %g e cada repetição a menos de %g de 0,5; a variação entre repetições é esperada)" % (REGRAS["embaralhamento_media"], REGRAS["embaralhamento_max"]) if (abs(m_ - 0.5) <= REGRAS["embaralhamento_media"] and d <= REGRAS["embaralhamento_max"]) else "ALERTA: longe de 0,5" for m_, d in zip(PERM_RES.roc_auc_medio, PERM_RES.max_desvio_abs)]
    print("\n== ROC-AUC na validação real, com o modelo treinado em rótulos embaralhados (esperado ~0,5) ==")
    display(PERM_RES.round(4))
else: print("Embaralhamento desligado (FAZER_PERM = False).")
T_SEC["4.1_embaralhamento"] = time.perf_counter() - t_sec0

Embaralhamento de rótulos: concluído em 3min 20s

== ROC-AUC na validação real, com o modelo treinado em rótulos embaralhados (esperado ~0,5) ==


,modelo,repeticoes,roc_auc_medio,roc_auc_min,roc_auc_max,max_desvio_abs,n_treino,n_avaliacao,situacao
0,RF ajustado,5,0.5042,0.4305,0.5767,0.0767,81228,60154,"ok (média a menos de 0.03 e cada repetição a menos de 0.1 de 0,5; a variação entre repetições é esperada)"
1,SVM ajustado,3,0.4743,0.4423,0.5113,0.0577,10002,20004,"ok (média a menos de 0.03 e cada repetição a menos de 0.1 de 0,5; a variação entre repetições é esperada)"


## 4.2 Linhas repetidas entre o ajuste e a validação
O embaralhamento (4.1) **não** detecta linhas repetidas entre o ajuste e a validação. Esta etapa conta, **diretamente e sem treinar nada**, as linhas da validação cujo vetor de atributos é **idêntico** ao de alguma linha do ajuste, e quantas delas têm rótulo **diferente** (conflito). Se a fração for relevante, a validação do mesmo dia é **otimista**, e a lacuna treino − validação subestima o sobreajuste.

In [11]:
# ---------- 4.2 Linhas repetidas entre o ajuste e a validação ----------
# O embaralhamento de rótulos (4.1) NÃO detecta linhas repetidas entre o ajuste e a validação (um modelo treinado em rótulos embaralhados não usa os rótulos reais). Esta contagem direta detecta: ela
# compara os vetores de atributos (exatamente, byte a byte) e mostra quantas linhas da validação têm um vetor idêntico a uma linha do ajuste, e quantas dessas têm RÓTULO diferente (conflito).
t_sec0 = time.perf_counter()
def _chaves(X):
    X = np.ascontiguousarray(X, dtype=np.float64)
    if X.shape[0] == 0: return []
    mv = memoryview(X).cast("B"); w = X.shape[1] * 8; return [hashlib.blake2b(mv[i * w:(i + 1) * w], digest_size=16).digest() for i in range(X.shape[0])]       # chave de 128 bits do vetor
rot_aj = {}
for k_, y_ in zip(_chaves(X_aj), y_aj): rot_aj.setdefault(k_, set()).add(int(y_))
ch_va = _chaves(X_va); dup = np.array([k_ in rot_aj for k_ in ch_va], bool); conf = np.array([(k_ in rot_aj) and (int(y_) not in rot_aj[k_]) for k_, y_ in zip(ch_va, y_va)], bool)
NOMES_OV, GVA_OV = TIPOS, g_va
linhas = [{"grupo": "todas", "linhas na validação": len(y_va), "com vetor idêntico ao de uma linha do ajuste": int(dup.sum()), "fração": float(dup.mean()), "idênticas com rótulo DIFERENTE (conflito)": int(conf.sum())}]
for gi, nome in enumerate(NOMES_OV):
    m = GVA_OV == gi
    if m.any(): linhas.append({"grupo": nome, "linhas na validação": int(m.sum()), "com vetor idêntico ao de uma linha do ajuste": int(dup[m].sum()), "fração": float(dup[m].mean()), "idênticas com rótulo DIFERENTE (conflito)": int(conf[m].sum())})
OVERLAP_TAB = pd.DataFrame(linhas).set_index("grupo"); OV_FRAC = float(dup.mean()); OV_SEV = sev(OV_FRAC, REGRAS["sobreposicao_validacao"])
OV_SEV_GRUPO = pior(*[sev(r["fração"], REGRAS["sobreposicao_validacao"]) for g_, r in OVERLAP_TAB.iterrows() if g_ != "todas" and r["linhas na validação"] >= 100])
OV_INFO = {"fracao_validacao_com_vetor_do_ajuste": OV_FRAC, "linhas": int(dup.sum()), "conflitos": int(conf.sum()), "gravidade_geral": OV_SEV, "pior_gravidade_por_grupo": OV_SEV_GRUPO, "regra": REGRAS["sobreposicao_validacao"]}
print("== Linhas da validação cujo vetor de atributos é IDÊNTICO ao de alguma linha do ajuste ==")
display(OVERLAP_TAB.round(4))
print(f"\nNo total, {100 * OV_FRAC:.2f}% das linhas da validação repetem um vetor do ajuste ({int(dup.sum()):,} de {len(y_va):,}); {int(conf.sum()):,} delas têm rótulo diferente. Gravidade geral: {OV_SEV}; pior grupo (com ao menos 100 linhas): {OV_SEV_GRUPO}.")
print("Se for relevante, a validação do mesmo dia é OTIMISTA (o modelo já viu aquele vetor), e a lacuna treino − validação subestima o sobreajuste." if OV_SEV in ("moderada", "grande") else "A validação é praticamente independente do ajuste quanto a vetores repetidos.")
T_SEC["4.2_sobreposicao"] = time.perf_counter() - t_sec0


== Linhas da validação cujo vetor de atributos é IDÊNTICO ao de alguma linha do ajuste ==


,linhas na validação,com vetor idêntico ao de uma linha do ajuste,fração,idênticas com rótulo DIFERENTE (conflito)
grupo,,,,
todas,60154,481,0.0080,4
BENIGN,10154,37,0.0036,0
DNS,4538,110,0.0242,3
LDAP,4538,96,0.0212,0
MSSQL,4538,50,0.0110,0
NTP,4538,0,0.0000,0
NetBIOS,4537,96,0.0212,0
SNMP,4538,62,0.0137,0
SSDP,4538,9,0.0020,0



No total, 0.80% das linhas da validação repetem um vetor do ajuste (481 de 60,154); 4 delas têm rótulo diferente. Gravidade geral: pequena; pior grupo (com ao menos 100 linhas): moderada.
A validação é praticamente independente do ajuste quanto a vetores repetidos.


## 5. Validação adversarial, classe a classe (os dois dias são diferentes?)
Para cada classe que existe **nos dois dias** (BENIGN e os tipos de ataque presentes no treino e no teste), sorteamos linhas **únicas** do treino e do teste e treinamos um RF (`ARVORES_ADV` árvores, profundidade 12) para tentar adivinhar **de qual dia** cada linha vem, com validação cruzada de 5 dobras:
- **ROC-AUC ≈ 0,5:** o classificador não distingue os dois dias; a classe se comporta igual nos dois;
- **ROC-AUC ≈ 1:** os dois dias são facilmente distinguíveis; um modelo treinado num dia vê, no outro, dados que nunca viu.

A separação **por classe** é essencial: comparar o treino inteiro com o teste inteiro só mediria que as classes têm proporções diferentes (o treino tem TFTP, NTP e SSDP; o teste, quase só `Syn`). O **controle** divide ao acaso as linhas do **treino** em dois grupos falsos de "dia": a AUC deve dar ~0,5 (se der mais, o método não é confiável e a checagem final reprova). Mostramos também os atributos que mais separam os dois dias e como a mediana deles muda.

Como o teste tem muitas linhas repetidas (~89%), as amostras são de linhas **únicas**; o pedido ao servidor é ampliado até obter linhas únicas suficientes (limite `LIMITE_PEDIDO` linhas por classe, para poupar memória). Classes só do teste (por exemplo, `Portmap`) ou só do treino não têm contraparte e ficam de fora.

In [12]:
# ---------- 5.1 Validação adversarial por classe ----------
t_sec0 = time.perf_counter(); ADV, ADV_FEAT, FIG_ADV, COMUNS, SO_TESTE = pd.DataFrame(), {}, None, [], []
LIMITE_PEDIDO = globals().get("LIMITE_PEDIDO", 300_000)
if FAZER_ADV:
    cols_sql = ", ".join("v." + q(c) for c in CM)
    def contar(tab, onde=""):
        d = consultar_num(f"SELECT label_multi AS tipo, count() AS n FROM {tab}{onde} GROUP BY label_multi"); d["tipo"] = d["tipo"].map(nome_tipo); d = d.groupby("tipo", as_index=False)["n"].sum(); return {r.tipo: int(r.n) for r in d.itertuples()}
    N_TREINO_TIPO, N_TESTE_TIPO = contar(TAB_TREINO), contar(TAB_TESTE, " WHERE sobrepoe_treino = 0")
    COMUNS = ["BENIGN"] + sorted(t for t in N_TESTE_TIPO if t != "BENIGN" and t in N_TREINO_TIPO); SO_TESTE = sorted(t for t in N_TESTE_TIPO if t not in N_TREINO_TIPO)
    def amostra(view, tab, classe, n, extra=""):
        ids = f"SELECT id_linha FROM {tab} WHERE label_multi IN ({lit(classe)}, {lit('DrDoS_' + classe)}){extra} ORDER BY cityHash64(id_linha, {int(SEMENTE)}) LIMIT {int(n)}"
        return ler_csv_com_espera(f"SELECT {cols_sql} FROM {view} AS v WHERE v.id_linha IN ({ids})", CM).drop_duplicates(ignore_index=True)
    def amostra_unica(view, tab, classe, alvo, total, extra=""):
        pedir = int(min(total, LIMITE_PEDIDO, max(alvo * 1.5, 1000)))
        for _ in range(4):
            df = amostra(view, tab, classe, pedir, extra)
            if len(df) >= alvo or pedir >= min(total, LIMITE_PEDIDO): break
            pedir = int(min(total, LIMITE_PEDIDO, pedir * min(max(alvo / max(len(df), 1), 1.5), 10)))
        return df.sample(frac=1, random_state=SEMENTE).reset_index(drop=True).head(alvo), pedir
    def rf_dom(semente): return RandomForestClassifier(ARVORES_ADV, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=semente)
    def auc_dominio(A, B, semente):
        X = np.vstack([A, B]); d = np.r_[np.zeros(len(A), int), np.ones(len(B), int)]; oof = np.zeros(len(d))
        for itr, ite in StratifiedKFold(K_ADV, shuffle=True, random_state=semente).split(X, d): oof[ite] = rf_dom(semente).fit(X[itr], d[itr]).predict_proba(X[ite])[:, 1]
        return float(roc_auc_score(d, oof)), X, d
    def recall_tipo_gravado(pasta):
        try:
            d = pd.read_csv(pasta / "recall_por_tipo_teste.csv"); return dict(zip(d.iloc[:, 0].map(nome_tipo), d.iloc[:, 3].astype(float)))
        except Exception: return {}
    REC_RF, REC_SVM = recall_tipo_gravado(PASTA_RF), recall_tipo_gravado(PASTA_SVM)
    barra = Barra(len(COMUNS), "Validação adversarial"); linhas = []
    for c in COMUNS:
        t0 = time.perf_counter()
        d_tr, p_tr = amostra_unica(VIEW_TREINO, TAB_TREINO, c, 2 * N_ADV, N_TREINO_TIPO[c]); d_te, p_te = amostra_unica(VIEW_TESTE, TAB_TESTE, c, N_ADV, N_TESTE_TIPO[c], " AND sobrepoe_treino = 0")
        m = min(len(d_tr), len(d_te)); linha = {"classe": c, "linhas_treino": N_TREINO_TIPO[c], "linhas_teste": N_TESTE_TIPO[c], "unicas_amostra_treino": len(d_tr), "unicas_amostra_teste": len(d_te), "usadas_por_dia": m,
                                                "recall_teste_RF_ajustado": REC_RF.get(c, np.nan), "recall_teste_SVM_ajustado": REC_SVM.get(c, np.nan)}
        if m < MIN_LINHAS_ADV:
            linha.update({"auc_adversarial": np.nan, "auc_controle": np.nan, "top_atributos": "(poucas linhas únicas: classe pulada)"}); linhas.append(linha); barra.avancar(1, f"{c}: pulada ({m} linhas únicas)"); continue
        A, B = d_tr.iloc[:m].to_numpy(float), d_te.iloc[:m].to_numpy(float)
        auc, X, d = auc_dominio(A, B, SEMENTE)
        mc = len(d_tr) // 2; Ac, Bc = d_tr.iloc[:mc].to_numpy(float), d_tr.iloc[mc:2 * mc].to_numpy(float)       # controle: duas metades do TREINO fingindo ser dois dias
        auc_c = auc_dominio(Ac, Bc, SEMENTE + 1)[0] if mc >= MIN_LINHAS_ADV else np.nan
        imp = rf_dom(SEMENTE).fit(X, d).feature_importances_; top = np.argsort(imp)[::-1][:5]
        linha.update({"auc_adversarial": auc, "auc_controle": auc_c, "top_atributos": ", ".join(CM[i] for i in top)})
        ADV_FEAT[c] = pd.DataFrame({"importancia": imp[top], **{f"treino_{n_}": np.percentile(A[:, top], p_, axis=0) for n_, p_ in (("p10", 10), ("p50", 50), ("p90", 90))}, **{f"teste_{n_}": np.percentile(B[:, top], p_, axis=0) for n_, p_ in (("p10", 10), ("p50", 50), ("p90", 90))}}, index=[CM[i] for i in top])
        linhas.append(linha); barra.avancar(1, f"{c}: AUC {auc:.3f} (controle {auc_c:.3f}) | {m:,} linhas por dia | {time.perf_counter() - t0:.0f} s"); del A, B, X, Ac, Bc
        gc.collect()
    barra.fechar(); ADV = pd.DataFrame(linhas)
    ADV["gravidade"] = [sev(a, REGRAS["auc_adversarial"]) if (np.isfinite(a) and u >= REGRAS["min_linhas_unicas_adversarial"]) else "n/d" for a, u in zip(ADV.auc_adversarial, ADV.unicas_amostra_teste)]
    print("\n== Validação adversarial por classe (ROC-AUC de distinguir treino × teste; controle = treino × treino, esperado ~0,5) ==")
    display(ADV.drop(columns=["top_atributos"]).round(4))
    if SO_TESTE: print(f"\nClasses só no teste (sem contraparte no treino, fora desta análise): {SO_TESTE}")
    for c, tab_f in ADV_FEAT.items():
        if ADV.set_index("classe").loc[c, "gravidade"] in ("moderada", "grande"):
            print(f"\n-- {c}: atributos que mais separam os dois dias (percentis 10/50/90 em cada dia) --"); display(tab_f.round(4))
    try:
        import matplotlib.pyplot as plt
        d_ok = ADV[ADV.auc_adversarial.notna()].reset_index(drop=True)
        if len(d_ok):
            FIG_ADV, ax = plt.subplots(figsize=(8, 0.6 * len(d_ok) + 1.8)); y_ = np.arange(len(d_ok))
            ax.barh(y_, d_ok.auc_adversarial, color="C0", label="treino × teste"); ax.plot(d_ok.auc_controle, y_, "kD", label="controle (treino × treino)")
            for v_ in REGRAS["auc_adversarial"]: ax.axvline(v_, color="gray", ls="--", lw=0.8)
            ax.axvline(0.5, color="gray", lw=0.8); ax.set_yticks(y_); ax.set_yticklabels(d_ok.classe); ax.set_xlim(0.4, 1.02); ax.invert_yaxis(); ax.set_xlabel("ROC-AUC do classificador de dia (0,5 = indistinguível)")
            ax.set_title("Validação adversarial por classe"); ax.legend(loc="lower right"); FIG_ADV.tight_layout(); plt.close(FIG_ADV)
    except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")
else: print("Validação adversarial desligada (FAZER_ADV = False).")
T_SEC["5.1_adversarial"] = time.perf_counter() - t_sec0

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...


Validação adversarial: concluído em 1min 57s

== Validação adversarial por classe (ROC-AUC de distinguir treino × teste; controle = treino × treino, esperado ~0,5) ==


,classe,linhas_treino,linhas_teste,unicas_amostra_treino,unicas_amostra_teste,usadas_por_dia,recall_teste_RF_ajustado,recall_teste_SVM_ajustado,auc_adversarial,auc_controle,gravidade
0,BENIGN,50768,56216,40000,20000,20000,0.9824,0.9884,0.9488,0.4995,grande
1,LDAP,30099,112491,30099,7377,7377,0.9992,0.9994,0.8404,0.4977,grande
2,MSSQL,207003,854242,40000,20000,20000,0.9998,0.9997,0.7222,0.5014,moderada
3,NetBIOS,19490,91175,19388,7507,7507,0.9993,0.9994,0.9633,0.4842,grande
4,Syn,155515,4887060,40000,20000,20000,0.9307,0.9949,1.0000,0.4992,grande
5,UDP,1075337,1214746,40000,20000,20000,0.9999,1.0000,0.7388,0.5010,moderada
6,UDPLag,89066,557,40000,300,300,0.8995,0.8815,0.9913,0.5036,grande



Classes só no teste (sem contraparte no treino, fora desta análise): ['Portmap']

-- BENIGN: atributos que mais separam os dois dias (percentis 10/50/90 em cada dia) --


,importancia,treino_p10,treino_p50,treino_p90,teste_p10,teste_p50,teste_p90
min_seg_size_forward,0.1155,20.0,20.0,3.200000e+01,20.0,20.0,3.200000e+01
init_win_bytes_forward,0.0550,-1.0,252.0,1.653700e+04,-1.0,255.0,2.920000e+04
fwd_header_length,0.0547,20.0,40.0,3.440000e+02,20.0,64.0,6.440000e+02
avg_fwd_segment_size,0.0532,0.0,31.0,8.897040e+01,0.0,31.0,1.050625e+02
flow_iat_max,0.0476,166.0,20984.0,9.990323e+06,60.0,23953.5,1.018237e+07



-- LDAP: atributos que mais separam os dois dias (percentis 10/50/90 em cada dia) --


... (7 outputs left out)


-- UDP: atributos que mais separam os dois dias (percentis 10/50/90 em cada dia) --


,importancia,treino_p10,treino_p50,treino_p90,teste_p10,teste_p50,teste_p90
fwd_header_length,0.3058,0.0000,80.0000,700.0000,0.0000,72.0000,144.0000
min_seg_size_forward,0.2783,-1.0000,8.0000,20.0000,-1.0000,8.0000,20.0000
flow_iat_min,0.0535,0.0000,1.0000,1.0000,0.0000,1.0000,1.0000
flow_iat_max,0.0511,105123.0000,109526.0000,110770.0000,104860.0000,109693.0000,110842.0000
flow_iat_std,0.0459,56378.8449,59812.9695,62859.5879,56360.7032,59590.7859,62837.3599



-- UDPLag: atributos que mais separam os dois dias (percentis 10/50/90 em cada dia) --


,importancia,treino_p10,treino_p50,treino_p90,teste_p10,teste_p50,teste_p90
flow_iat_std,0.1204,58713.597,7.046196e+06,1.226574e+07,0.0000,603.5305,390656.3312
min_seg_size_forward,0.1065,-0.100,2.000000e+01,2.000000e+01,20.0000,32.0000,32.0000
fwd_packets_s,0.0732,0.151,2.330000e-01,3.767310e+01,2.3063,1124.3386,666666.6667
flow_iat_mean,0.0718,35392.200,4.358120e+06,7.803122e+06,2.0000,442.5000,216031.2576
flow_iat_min,0.0604,0.000,1.000000e+00,1.000000e+00,1.0000,2.0000,4.0000


## 6. Veredito (pelas regras fixadas na introdução)
Para cada modelo: **Q1** (decorar o treino) e **Q2** (generalizar para outro dia), com a pior gravidade entre as evidências de cada pergunta, e a **leitura** correspondente. O texto final é uma **sugestão** para o TCC: ele diz o que os números mostram e o que **não** provam.

In [13]:
# ---------- 6.1 Veredito ----------
t_sec0 = time.perf_counter()
pt = lambda x, nd=2: f"{x:.{nd}f}".replace(".", ",")                       # número com vírgula decimal (para o texto do TCC)
if len(PERM_RES): perm_sit = {r.modelo: ("pequena" if r.situacao.startswith("ok") else "grande") for r in PERM_RES.itertuples()}
else: perm_sit = {}
adv_validas = ADV[ADV.gravidade.isin(["pequena", "moderada", "grande"])] if len(ADV) else pd.DataFrame()
if len(adv_validas): adv_pior, adv_cls = pior(*adv_validas.gravidade), adv_validas.sort_values("auc_adversarial", ascending=False).iloc[0]
else: adv_pior, adv_cls = "n/d", None
VEREDITO = {}
for chave, M in MODELOS.items():
    rot = M["rotulo"]; L = LAC.loc[rot]
    q1 = pior(L["Q1 acc.bal."], L["Q1 ROC-AUC"], perm_sit.get(rot, "n/d")); q2_modelo = pior(L["Q2 ROC-AUC"], L["Q2 falso alarme"], L["Q2 recall"]); q2 = "n/d" if q2_modelo == "n/d" else pior(q2_modelo, adv_pior)
    lacuna = f"lacuna treino−validação: acurácia balanceada {TAB.loc[rot, 'lacuna_acc_bal']:+.4f}, ROC-AUC {TAB.loc[rot, 'lacuna_roc_auc']:+.5f}"
    queda = f"validação→teste: ROC-AUC {-TAB.loc[rot, 'queda_roc_auc_teste']:+.5f}, falso alarme ×{TAB.loc[rot, 'razao_falso_alarme']:.1f}, recall {-TAB.loc[rot, 'queda_recall']:+.4f}"
    if q1 in ("moderada", "grande"): leitura = "HÁ SINAL de sobreajuste clássico (decorar o treino): " + lacuna + (" e o embaralhamento de rótulos fugiu de 0,5 (média das repetições fora de ±0,03 ou repetição fora de ±0,10: investigar)" if perm_sit.get(rot) == "grande" else "")
    elif q2_modelo == "n/d": leitura = "sem sinal de decorar o treino; a comparação com o teste não pôde ser feita (arquivos do teste ausentes)"
    elif q2 == "pequena": leitura = "SEM SINAL de sobreajuste de nenhum tipo: " + lacuna + "; " + queda
    elif adv_pior in ("moderada", "grande") and q2_modelo in ("moderada", "grande"): leitura = f"sem sinal de decorar o treino ({lacuna}); a queda no teste ({queda}) é COMPATÍVEL com mudança de distribuição entre os dias (classe mais distinta: {adv_cls['classe']}, ROC-AUC adversarial {adv_cls['auc_adversarial']:.3f})"
    elif q2_modelo in ("moderada", "grande"): leitura = f"sem sinal de decorar o treino ({lacuna}); há queda no teste ({queda}), mas a validação adversarial NÃO mostra mudança relevante nos atributos: a causa precisa de outra investigação"
    else: leitura = f"sem sinal de decorar o treino ({lacuna}); queda no teste pequena; a validação adversarial indica mudança entre os dias (ROC-AUC {adv_cls['auc_adversarial']:.3f} em {adv_cls['classe']}), sem efeito relevante neste modelo" if adv_cls is not None else "sem sinal"
    VEREDITO[rot] = {"Q1_decorar_o_treino": q1, "Q2_generalizar_outro_dia": q2, "Q2_so_modelo": q2_modelo, "leitura": leitura}
    ganho = GANHO_50_100.get(rot); VEREDITO[rot]["curva_ganho_50_100"] = None if ganho is None else {"ganho_roc_auc": ganho, "situacao": sev(ganho, REGRAS["curva_ganho_50_100"])}
RES = pd.DataFrame({r: {"Q1 decorar o treino": v["Q1_decorar_o_treino"], "Q2 outro dia": v["Q2_generalizar_outro_dia"]} for r, v in VEREDITO.items()}).T
print("== Gravidade dos sinais, por modelo ==\n"); display(RES)
for r, v in VEREDITO.items(): print(f"• {r}: {v['leitura']}")
print("\n--- Texto sugerido para o TCC (revise antes de usar) ---")
n_ok = sum(1 for v in VEREDITO.values() if v["Q1_decorar_o_treino"] == "pequena")
TEXTO = (f"Para investigar sobreajuste, comparamos o desempenho dos {len(MODELOS)} modelos binários nas próprias linhas de treino, na validação cruzada, em uma validação separada do mesmo dia e no teste de outro dia, "
         f"e complementamos com curvas de aprendizagem, um controle com rótulos embaralhados e uma validação adversarial por classe (treino × teste). "
         + ("Nenhum dos modelos apresentou sinal de memorização do treino segundo as regras definidas antes da análise (lacuna treino−validação pequena e controle com rótulos embaralhados em torno de 0,5). " if n_ok == len(VEREDITO) else "Houve sinal de memorização do treino em pelo menos um modelo segundo as regras definidas antes da análise (ver a tabela de gravidade). ")
         + (f"{pt(100 * OV_FRAC, 2)}% das linhas da validação têm vetor de atributos idêntico ao de uma linha do ajuste ({pt(OV_INFO['linhas'], 0)} linhas; {pt(OV_INFO['conflitos'], 0)} com rótulo diferente)" + (", o que torna a validação do mesmo dia otimista. " if OV_SEV in ("moderada", "grande") else ", ou seja, a validação é praticamente independente do ajuste quanto a vetores repetidos. "))
         + (f"A queda de desempenho no teste (de outro dia) é compatível com mudança de distribuição entre os dias: a validação adversarial distinguiu treino e teste com ROC-AUC de até {pt(adv_cls['auc_adversarial'])} (classe {adv_cls['classe']}), enquanto o controle do mesmo dia ficou em torno de 0,5. " if adv_cls is not None and adv_pior in ("moderada", "grande") else "")
         + ("Como a validação do mesmo dia já está perto do teto (ROC-AUC acima de 0,999), a lacuna treino−validação tem pouco espaço para revelar sobreajuste; por isso o diagnóstico se apoia também na curva de aprendizagem, no controle com rótulos embaralhados e na validação adversarial. " if float(TAB.validacao_roc_auc.max()) > 0.999 else "")
         + "Esses resultados são evidências descritivas, obtidas em uma única partição de teste com muitas linhas repetidas; não provam a ausência de sobreajuste, e a validação do mesmo dia do treino não é capaz de detectar o ajuste às condições de um dia específico.")
print(TEXTO)
T_SEC["6.1_veredito"] = time.perf_counter() - t_sec0

== Gravidade dos sinais, por modelo ==



,Q1 decorar o treino,Q2 outro dia
RF ajustado,pequena,grande
RF padrão,pequena,grande
SVM ajustado,pequena,grande
SVM padrão,pequena,grande


• RF ajustado: sem sinal de decorar o treino (lacuna treino−validação: acurácia balanceada +0.0007, ROC-AUC +0.00001); a queda no teste (validação→teste: ROC-AUC -0.00864, falso alarme ×22.4, recall -0.0466) é COMPATÍVEL com mudança de distribuição entre os dias (classe mais distinta: Syn, ROC-AUC adversarial 1.000)
• RF padrão: sem sinal de decorar o treino (lacuna treino−validação: acurácia balanceada +0.0008, ROC-AUC +0.00006); a queda no teste (validação→teste: ROC-AUC -0.01000, falso alarme ×26.6, recall -0.0481) é COMPATÍVEL com mudança de distribuição entre os dias (classe mais distinta: Syn, ROC-AUC adversarial 1.000)
• SVM ajustado: sem sinal de decorar o treino (lacuna treino−validação: acurácia balanceada -0.0007, ROC-AUC +0.00013); a queda no teste (validação→teste: ROC-AUC -0.00418, falso alarme ×5.1, recall +0.0002) é COMPATÍVEL com mudança de distribuição entre os dias (classe mais distinta: Syn, ROC-AUC adversarial 1.000)
• SVM padrão: sem sinal de decorar o treino (lac

## 7. Gravação
Tudo vai para `artefatos_modelos/diagnostico_sobreajuste/diag_<data>/`: as tabelas (CSV), os gráficos (PNG), o veredito com o texto sugerido (`veredito.json`), as regras de decisão (gravadas **no início** do notebook, antes de qualquer resultado), a configuração, o tempo de cada etapa e uma linha em `registro_diagnosticos.csv`. Nenhum arquivo dos modelos é alterado.

In [14]:
# ---------- 7.1 Gravação ----------
t_sec0 = time.perf_counter()
TAB.reset_index().to_csv(PASTA_DIAG / "tabela_treino_cv_validacao_teste.csv", index=False); LAC.reset_index().to_csv(PASTA_DIAG / "lacunas_e_gravidades.csv", index=False)
if len(CURVA): CURVA.to_csv(PASTA_DIAG / "curva_aprendizagem_sorteios.csv", index=False); CURVA_RES.to_csv(PASTA_DIAG / "curva_aprendizagem.csv", index=False)
if len(PERM): PERM.to_csv(PASTA_DIAG / "embaralhamento_rotulos_repeticoes.csv", index=False); PERM_RES.to_csv(PASTA_DIAG / "embaralhamento_rotulos.csv", index=False)
OVERLAP_TAB.reset_index().to_csv(PASTA_DIAG / "sobreposicao_ajuste_validacao.csv", index=False)
if len(ADV):
    ADV.to_csv(PASTA_DIAG / "validacao_adversarial_por_classe.csv", index=False)
    if ADV_FEAT: pd.concat(ADV_FEAT, names=["classe", "atributo"]).reset_index().to_csv(PASTA_DIAG / "validacao_adversarial_atributos.csv", index=False)
for fig, nome in ((FIG_CURVA, "curva_aprendizagem.png"), (FIG_ADV, "validacao_adversarial.png")):
    if fig is not None: fig.savefig(PASTA_DIAG / nome, dpi=130, bbox_inches="tight")
T_SEC["7.1_gravacao"] = time.perf_counter() - t_sec0
pd.DataFrame({"etapa": list(T_SEC), "segundos": [round(v, 1) for v in T_SEC.values()]}).to_csv(PASTA_DIAG / "tempos_diagnostico.csv", index=False)
TEMPO_TOTAL = float(sum(T_SEC.values()))
salvar_json({"veredito": VEREDITO, "sobreposicao_ajuste_validacao": OV_INFO, "texto_sugerido_tcc": TEXTO, "tempo_total_s": TEMPO_TOTAL, "tempos_por_etapa_s": T_SEC}, PASTA_DIAG / "veredito.json")
salvar_json({"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "origem": "notebook 13 (diagnóstico de sobreajuste)", "semente": SEMENTE,
             "parametros": {"tamanhos_curva": list(TAMANHOS_CURVA), "reps_curva": REPS_CURVA, "curva_padrao": CURVA_PADRAO, "n_perm_rf": N_PERM_RF, "n_perm_svm": N_PERM_SVM, "n_svm_perm": N_SVM_PERM, "n_val_perm_svm": N_VAL_PERM_SVM,
                            "n_adv": N_ADV, "arvores_adv": ARVORES_ADV, "k_adv": K_ADV, "limite_pedido": LIMITE_PEDIDO if "LIMITE_PEDIDO" in globals() else None, "fazer": {"curva": FAZER_CURVA, "embaralhamento": FAZER_PERM, "adversarial": FAZER_ADV}},
             "pastas": {"rf_ajustado": str(PASTA_RF), "rf_padrao": str(PASTA_RF_PAD) if PASTA_RF_PAD else None, "svm": str(PASTA_SVM)}, "versao_selecao_notebook04": versao_sel, "versao_views_notebook03": versao_views,
             "dados": {"ajuste": len(X_aj), "validacao": len(X_va), "atributos": len(CM)}, "modelos": {k: {"rotulo": M["rotulo"], "limiar_natural": M["limiar"], "descricao": M["descricao"]} for k, M in MODELOS.items()},
             "classes_comuns_treino_teste": COMUNS, "classes_so_no_teste": SO_TESTE, "software": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": __import__("sklearn").__version__}}, PASTA_DIAG / "config_diagnostico.json")
reg = {"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "tempo_total_s": round(TEMPO_TOTAL, 1), **{f"{r}_Q1": v["Q1_decorar_o_treino"] for r, v in VEREDITO.items()}, **{f"{r}_Q2": v["Q2_generalizar_outro_dia"] for r, v in VEREDITO.items()}, "pasta": str(PASTA_DIAG)}
reg_csv = PASTA_DIAG.parent / "registro_diagnosticos.csv"; novo = pd.DataFrame([reg])
if reg_csv.exists(): novo = pd.concat([pd.read_csv(reg_csv), novo], ignore_index=True)
novo.to_csv(reg_csv, index=False)
print(f"✔ Gravado em {PASTA_DIAG}")
print(f"\n⏱ Tempo do diagnóstico: {fmt_tempo(TEMPO_TOTAL)} no total | " + " | ".join(f"{k.split('_', 1)[1]} {fmt_tempo(v)}" for k, v in T_SEC.items() if v > 1))

✔ Gravado em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\diagnostico_sobreajuste\diag_20261111_230952

⏱ Tempo do diagnóstico: 20min 55s no total | tabela 58s | curva 14min 31s | embaralhamento 3min 20s | adversarial 2min 03s | gravacao 1s


## 8. Conferências
Confere que o diagnóstico é confiável: os modelos reproduzem os números que a busca gravou, a curva de 100% reproduz o modelo original, as regras foram gravadas antes dos resultados, o controle do mesmo dia da validação adversarial dá ~0,5 (se não desse, o método não mediria mudança de dia) e os arquivos foram gravados.

In [15]:
# ---------- 8.1 Conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
dif_v = [M["rotulo"] for M in MODELOS.values() if any(M["cfg"].get(k) not in (None, v) for k, v in (("versao_selecao_notebook04", versao_sel), ("versao_views_notebook03", versao_views)))]
verifica("Versões da seleção (notebook 04) e das views (notebook 03) iguais às gravadas em cada modelo", not dif_v, f"({versao_sel} | {versao_views})" if not dif_v else f"(diferem: {dif_v}; esses modelos não foram treinados com os dados atuais: refaça-os ou exclua-os com INCLUIR_RF_PADRAO = False)")
dad = [(M["rotulo"], M["cfg"].get("dados", {})) for M in MODELOS.values() if M["cfg"].get("dados")]
verifica("Ajuste e validação têm os tamanhos gravados pelos notebooks de treino", all(d.get("ajuste") in (None, len(X_aj)) and d.get("validacao") in (None, len(X_va)) for _, d in dad), f"({len(X_aj):,} e {len(X_va):,})")
verifica("Ajuste e validação são disjuntos e o tipo BENIGN coincide com y = 0", not set(aj["id_linha"]) & set(va["id_linha"]) and ((g_aj == 0) == (y_aj == 0)).all() and ((g_va == 0) == (y_va == 0)).all())
verifica("Todos os modelos aceitam os atributos das views (mesmo número e ordem de colunas)", all(getattr(M["modelo"], "n_features_in_", len(CM)) == len(CM) for M in MODELOS.values()), f"({len(CM)} atributos)")
verifica("Escores finitos em todos os modelos", all(np.isfinite(e["aj"]).all() and np.isfinite(e["va"]).all() for e in ESC.values()))
def roc_gravado(M):
    try:
        if M["cv_fonte"][0] == "json": return float(json.loads((M["pasta"] / "metricas_validacao.json").read_text(encoding="utf-8"))["sem_pesos"]["roc_auc"])
        g = pd.read_csv(M["cv_fonte"][1] / "resultados_grade.csv"); return float(g[g["config"] == M["cv_fonte"][2]].iloc[0]["val_roc_auc"])
    except Exception: return np.nan
difs = {M["rotulo"]: abs(TAB.loc[M["rotulo"], "validacao_roc_auc"] - roc_gravado(M)) for M in MODELOS.values()}; difs_ok = {k: v for k, v in difs.items() if np.isfinite(v)}
verifica("Os modelos reproduzem o ROC-AUC de validação gravado pelos notebooks de treino", bool(difs_ok) and max(difs_ok.values()) < 1e-4, f"(maior diferença {max(difs_ok.values()):.2e} em {len(difs_ok)} modelo(s))" if difs_ok else "(nenhum valor gravado encontrado)")
if len(CURVA_RES):
    d1 = CURVA_RES[CURVA_RES.fracao >= 1.0]; dif_c = max(abs(r.validacao_roc_auc - TAB.loc[r.modelo, "validacao_roc_auc"]) for r in d1.itertuples())
    verifica("A curva de aprendizagem em 100% reproduz o modelo original (mesmo ROC-AUC na validação)", dif_c < 1e-6, f"(diferença {dif_c:.1e})")
    verifica("Toda subamostra da curva tem todos os tipos de ataque", all(len(np.unique(g_aj[sub_estrat(g_aj, f, SEMENTE)])) == len(TIPOS) for f in TAMANHOS_CURVA), f"({len(TIPOS) - 1} tipos)")
verifica("Sobreposição ajuste × validação: as contagens fecham (idênticas ≤ linhas; conflitos ≤ idênticas; soma dos grupos = total)", int(OVERLAP_TAB.loc["todas", "com vetor idêntico ao de uma linha do ajuste"]) <= len(y_va) and OV_INFO["conflitos"] <= OV_INFO["linhas"] and int(OVERLAP_TAB.drop(index="todas")["com vetor idêntico ao de uma linha do ajuste"].sum()) == OV_INFO["linhas"], f"({OV_INFO['linhas']:,} de {len(y_va):,})")
if len(PERM): verifica("Embaralhamento: os rótulos foram de fato embaralhados (concordância com os reais perto de 50%)", float(PERM.concordancia_com_rotulo_real.max()) < 0.6, f"(máx. {PERM.concordancia_com_rotulo_real.max():.3f})")
if len(ADV):
    ctrl = ADV.auc_controle.dropna(); verifica("Validação adversarial: o controle do mesmo dia (treino × treino) dá ~0,5 em toda classe", len(ctrl) > 0 and ctrl.between(0.45, 0.55).all(), f"(controles entre {ctrl.min():.3f} e {ctrl.max():.3f})" if len(ctrl) else "(sem classes com linhas suficientes)")
    verifica("Validação adversarial: há ao menos um tipo de ataque em comum entre treino e teste, além do BENIGN", len(COMUNS) >= 2, f"({COMUNS})")
verifica("As regras de decisão foram gravadas antes dos resultados", (PASTA_DIAG / "regras_decisao.json").stat().st_mtime <= (PASTA_DIAG / "tabela_treino_cv_validacao_teste.csv").stat().st_mtime)
esperados = ["regras_decisao.json", "tabela_treino_cv_validacao_teste.csv", "lacunas_e_gravidades.csv", "veredito.json", "config_diagnostico.json", "tempos_diagnostico.csv"] + (["curva_aprendizagem.csv"] if len(CURVA) else []) + (["embaralhamento_rotulos.csv"] if len(PERM) else []) + (["validacao_adversarial_por_classe.csv"] if len(ADV) else [])
verifica("Arquivos esperados gravados", all((PASTA_DIAG / f).exists() for f in esperados), f"({len(esperados)} arquivos)")
assert all(ok), "há checagens reprovadas acima"
(PASTA_DIAG / "FINALIZADO").write_text(datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_DIAG}")
print("\nPróximo passo: me mande o texto desta saída e os arquivos tabela_treino_cv_validacao_teste.csv, curva_aprendizagem.csv, embaralhamento_rotulos.csv, validacao_adversarial_por_classe.csv, veredito.json e os PNG.")

✔ Versões da seleção (notebook 04) e das views (notebook 03) iguais às gravadas em cada modelo (20261018_204854 | 20261018_180729)
✔ Ajuste e validação têm os tamanhos gravados pelos notebooks de treino (81,228 e 60,154)
✔ Ajuste e validação são disjuntos e o tipo BENIGN coincide com y = 0 
✔ Todos os modelos aceitam os atributos das views (mesmo número e ordem de colunas) (43 atributos)
✔ Escores finitos em todos os modelos 
✔ Os modelos reproduzem o ROC-AUC de validação gravado pelos notebooks de treino (maior diferença 7.88e-09 em 4 modelo(s))
✔ A curva de aprendizagem em 100% reproduz o modelo original (mesmo ROC-AUC na validação) (diferença 1.1e-16)
✔ Toda subamostra da curva tem todos os tipos de ataque (12 tipos)
✔ Sobreposição ajuste × validação: as contagens fecham (idênticas ≤ linhas; conflitos ≤ idênticas; soma dos grupos = total) (481 de 60,154)
✔ Embaralhamento: os rótulos foram de fato embaralhados (concordância com os reais perto de 50%) (máx. 0.502)
✔ Validação adversar